In [1]:
import os
base_path = os.getcwd()

In [2]:
import polars as pl

local_parquet_path = os.path.join(base_path, "data", "train.parquet")

# Menggunakan scan_parquet (Lazy) agar RAM tetap hemat
# Data hanya diproses saat .collect() dipanggil
df_pq = pl.scan_parquet(local_parquet_path)

In [3]:
df_sample = df_pq.head(15_000_000).collect()

print(f"Data berhasil disimpan di variabel 'df_sample'. Jumlah baris: {len(df_sample)}")
display(df_sample)

Data berhasil disimpan di variabel 'df_sample'. Jumlah baris: 15000000


ip,app,device,os,channel,click_time,attributed_time,is_attributed
i64,i64,i64,i64,i64,str,str,i64
83230,3,1,13,379,"""2017-11-06 14:32:21""",null,0
17357,3,1,19,379,"""2017-11-06 14:33:34""",null,0
35810,3,1,13,379,"""2017-11-06 14:34:12""",null,0
45745,14,1,13,478,"""2017-11-06 14:34:52""",null,0
161007,3,1,13,379,"""2017-11-06 14:35:08""",null,0
…,…,…,…,…,…,…,…
85065,12,2,40,265,"""2017-11-07 01:37:13""",null,0
113719,3,1,31,424,"""2017-11-07 01:37:13""",null,0
43547,3,1,13,480,"""2017-11-07 01:37:13""",null,0


In [4]:
from feature_engineering import generate_fraud_features, apply_heuristic_rules
df_sample = generate_fraud_features(df_sample)

In [6]:
df_sample = apply_heuristic_rules(df_sample)

In [7]:
df_sample[ "rule_based_reason"].value_counts()

rule_based_reason,count
str,u32
"""Click Injection""",40
"""Burst Clicks""",564227
"""Super Human Speed""",3571265
"""Channel Spraying""",7716331
"""Persistent Emulator""",472855
"""Normal""",2675282


In [8]:
analisis_injeksi = (
    df_sample
    .filter(pl.col("rule_based_reason") == "Click Injection")
    .group_by("channel")
    .agg([
        pl.len().alias("jumlah_injeksi"),
        pl.col("ip").n_unique().alias("jumlah_ip_korban")
    ])
    .sort("jumlah_injeksi", descending=True)
)

print(analisis_injeksi)

shape: (13, 3)
┌─────────┬────────────────┬──────────────────┐
│ channel ┆ jumlah_injeksi ┆ jumlah_ip_korban │
│ ---     ┆ ---            ┆ ---              │
│ i64     ┆ u32            ┆ u32              │
╞═════════╪════════════════╪══════════════════╡
│ 113     ┆ 11             ┆ 11               │
│ 274     ┆ 9              ┆ 9                │
│ 21      ┆ 7              ┆ 7                │
│ 347     ┆ 3              ┆ 2                │
│ 234     ┆ 2              ┆ 2                │
│ …       ┆ …              ┆ …                │
│ 379     ┆ 1              ┆ 1                │
│ 215     ┆ 1              ┆ 1                │
│ 280     ┆ 1              ┆ 1                │
│ 424     ┆ 1              ┆ 1                │
│ 134     ┆ 1              ┆ 1                │
└─────────┴────────────────┴──────────────────┘


In [9]:
conversion_rates = (
    df_sample
    .group_by("rule_based_reason")
    .agg([
        pl.col("is_attributed").mean().alias("conversion_rate"),
        pl.len().alias("count")
    ])
    .sort("conversion_rate", descending=True)
)

print(conversion_rates)

shape: (6, 3)
┌─────────────────────┬─────────────────┬─────────┐
│ rule_based_reason   ┆ conversion_rate ┆ count   │
│ ---                 ┆ ---             ┆ ---     │
│ str                 ┆ f64             ┆ u32     │
╞═════════════════════╪═════════════════╪═════════╡
│ Click Injection     ┆ 1.0             ┆ 40      │
│ Normal              ┆ 0.006689        ┆ 2675282 │
│ Channel Spraying    ┆ 0.001496        ┆ 7716331 │
│ Burst Clicks        ┆ 0.000934        ┆ 564227  │
│ Persistent Emulator ┆ 0.000791        ┆ 472855  │
│ Super Human Speed   ┆ 0.00047         ┆ 3571265 │
└─────────────────────┴─────────────────┴─────────┘


In [10]:
from blocking_strategy import apply_business_blocking_strategy
df_final = apply_business_blocking_strategy(df_sample)

In [14]:
business_impact = (
    df_final
    .group_by("business_action")
    .agg([
        pl.len().alias("total_clicks"),
        pl.col("is_attributed").sum().alias("conversions_inside_action")
    ])
    .sort("business_action")
)
business_impact = (
    df_final
    .group_by("business_action")
    .agg([
        pl.len().alias("total_clicks"),
        pl.col("is_attributed").sum().alias("conversions_inside_action"),
        (pl.col("is_attributed").mean() * 100).alias("conversion_rate_percent")
    ])
    .sort("business_action")
)

print(business_impact)

shape: (3, 4)
┌──────────────────────────┬──────────────┬───────────────────────────┬─────────────────────────┐
│ business_action          ┆ total_clicks ┆ conversions_inside_action ┆ conversion_rate_percent │
│ ---                      ┆ ---          ┆ ---                       ┆ ---                     │
│ str                      ┆ u32          ┆ i64                       ┆ f64                     │
╞══════════════════════════╪══════════════╪═══════════════════════════╪═════════════════════════╡
│ FLAG_FOR_ML              ┆ 7716331      ┆ 11545                     ┆ 0.149618                │
│ HARD_BLOCK_REJECT_PAYOUT ┆ 4608387      ┆ 2619                      ┆ 0.056831                │
│ PASS_ORGANIC             ┆ 2675282      ┆ 17896                     ┆ 0.668939                │
└──────────────────────────┴──────────────┴───────────────────────────┴─────────────────────────┘
